# Tutorial: a protein and two disease variants

This notebook is the *hxex* tutorial, step by step. The same steps, with more explanation, are
in the user guide: [Tutorial](https://github.com/tuttlelm/hxex/blob/main/docs/guide/tutorial.md).

**How to use it.** Click a cell and press **Shift + Enter** to run it; the next cell is then
selected. Run the cells in order, from the top. A line starting with `!` runs an `hxex`
command, exactly as you would type it in a terminal. Running a cell again is safe: the cells
that change `hxex.toml` find the change already made, and `hxex init` and `hxex run` refuse to
replace what they wrote before (and say so).

**The data** are from the HDX-MS experiments of: Woods CN, Janowska MK, Ulmer LD, Kaur Sidhu J,
Stone NL, James EI, Guttman M, Bush MF, Klevit RE. Activation mechanism of small heat shock
protein HSPB5 revealed by disease-associated mutants. *Proc Natl Acad Sci USA* 122(20),
e2425061122 (2025). https://doi.org/10.1073/pnas.2425061122. Please cite it if you use the data.

HSPB5 (wild type, D109H and R120G) was labelled at room temperature in 85 % D₂O buffer, pH 7.5,
for 4 s, 1 min, 30 min, and 20 h, with undeuterated and fully deuterated controls. The spectra,
exported from HDExaminer, are in the `SpecExport` folder next to this notebook. The folders also
hold two quench-flow time points (0.1 s and 0.8 s) that are not part of the paper; step 4 leaves
them out.

## Helpers

Run this cell first. It defines three small helpers that the cells below use to change the
project file and to fill in the review table. You do not need to read it: the cells that use
it print every line they change, and the guide explains how to make the same changes in a text
editor instead.

In [ ]:
%matplotlib inline
import csv
from pathlib import Path

from IPython.display import Image

import hxex
from hxex.report import format_table, plot_controls_overview, series_table


def change(old, new, section=None):
    """Replace the start of the first line of hxex.toml that starts with `old` by `new`
    (after the line [section], or # [section], when given), and print the changed line. A
    line that already starts with `new` (the cell was run before) is left as it is."""
    path = Path("hxex.toml")
    lines = path.read_text().splitlines(keepends=True)
    start = 0
    if section is not None:
        start = next(i for i, line in enumerate(lines)
                     if line.strip().lstrip("#").strip() == section)
    for i in range(start, len(lines)):
        if lines[i].startswith(new):  # already changed
            print(lines[i].rstrip())
            return
        if lines[i].startswith(old):
            lines[i] = new + lines[i][len(old):]
            path.write_text("".join(lines))
            print(lines[i].rstrip())
            return
    raise ValueError(f"no line starting with {old!r}")


def add_before(line, text):
    """Add `text` to hxex.toml just above the line `line`, and print it (unless it is there
    already: the cell was run before)."""
    path = Path("hxex.toml")
    if text.strip() in path.read_text():
        print(text.strip())
        return
    lines = path.read_text().splitlines(keepends=True)
    i = next(i for i, x in enumerate(lines) if x.rstrip() == line)
    lines.insert(i, text.strip() + "\n\n")
    path.write_text("".join(lines))
    print(text.strip())


def review(sample, peptide_range, time, rep, review_K="", review_status="", note=""):
    """Fill in the review of one spectrum in user_review_table.csv (copied from
    results_tables/spectra.csv the first time)."""
    path = Path("user_review_table.csv")
    source = path if path.exists() else Path("results_tables/spectra.csv")
    with open(source, newline="") as f:
        rows = list(csv.DictReader(f))
    hits = [r for r in rows if r["sample"] == sample and r["peptide_range"] == peptide_range
            and float(r["time"] or "nan") == time and r["rep"] == str(rep)]
    for r in hits:
        r.update(review_K=str(review_K), review_status=review_status, note=note)
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    print(f"{len(hits)} spectrum(s) of {sample} {peptide_range}, {time} s, replicate {rep}: "
          f"review_K {review_K or '-'}, review_status {review_status or '-'}, note {note!r}")

## Step 3: write the project file

`hxex init` writes `hxex.toml`, the project file, with every setting at its default value. You can open it from Jupyter's file list (the browser tab you started in if running in a browser) to read it. Its `data_dir` is
already `"SpecExport"`, the folder next to it.

In [ ]:
!hxex init --format specexport

## Step 4: specify the hxex input parameters

A name for the project, two rules that leave out the quench-flow time points (the spectra are
kept, but not fitted), and the back-exchange correction with the labelling buffer's 85 % D₂O.

In [ ]:
change('project = "my_project"', 'project = "hspb5"')
add_before("[output]", """
[[input.exclude]]
time = "0.1 s"
note = "quench-flow time point (not in the paper)"

[[input.exclude]]
time = "0.8 s"
note = "quench-flow time point (not in the paper)"
""")
change("# [run.controls]", "[run.controls]", section="[run.controls]")
change("# d2o_fraction = 0.9", "d2o_fraction = 0.85", section="[run.controls]")

## Step 5: check

`hxex check` reads the settings and looks through the folders without analysing anything. It
ends with `ok`.

In [ ]:
!hxex check hxex.toml

## Step 6: run

The analysis itself: the spectra are read, the quench-flow ones left out, the isotope peaks
picked, and every spectrum fitted with one, two, and three populations. About two minutes on a
laptop. It writes the bundle (`results`) and the tables (`results_tables`).

In [ ]:
!hxex run hxex.toml

## Step 7: the tables

The tables are CSV files in the `results_tables` directory; open them from Jupyter's file list, or in a spreadsheet program. `series.csv` has a row for each peptide and charge in each sample:

In [ ]:
results = hxex.load_results("hxex.toml")
print(format_table(series_table(results), limit=10))

## Step 8: the figures

Three peptides of WT, at charge 2: residues 87-96 (one population), 42-54 (two populations at
the early times) and 68-75 (a weak spectrum at 30 min). In `series.csv` their `data_id` is 140,
56, and 107. The `[figures]` part of the project file says which to draw as files, and on which
time scale:

In [ ]:
change('series = "flagged"', "series = [140, 56, 107]", section="[figures]")
change('scale = "linear"', 'scale = "log"', section="[figures]")

In [ ]:
!hxex report hxex.toml --figures

In the notebook you can also draw any peptide straight away, by sample, residues, and charge.
One population at every time:

In [ ]:
results = hxex.load_results("hxex.toml")  # again, with the new [figures] settings
hxex.show_series(results, "WT", "87-96", 2, kinds=["spectra"])

Two populations at 4 s and 1 min, merging into one by 20 h; the controls (UN, FD) have one each:

In [ ]:
hxex.show_series(results, "WT", "42-54", 2)

**Try it:** change the sample (`"WT"`, `"D109H"`, or `"R120G"`), the residues, or the charge, and run the cell again. Add `k=1` or `k=2` to draw every spectrum with that number of populations,
or `format="svg"` for figures that stay sharp when you zoom in (slower for a large spectra page).

In [ ]:
hxex.show_series(results, "R120G", "42-54", 2, kinds=["uptake"])

The controls page of the overview: the retention of every fully deuterated control (the share of
the label it still held when measured; about 0.5 here, against 0.85 in the buffer).

In [ ]:
plot_controls_overview(results)

## Step 9: review

In WT 42-54 the 30 min replicates 1 and 3 have a tiny second population: one population is
enough. In WT 68-75 the 30 min replicate 2 is weak (flagged `low_intensity`). These decisions
go in the review table, a copy of `spectra.csv` with three columns filled in: `review_K`,
`review_status`, and `note`.

In [ ]:
review("WT", "0042-0054", 1800, 1, review_K=1, note="second population too small")
review("WT", "0042-0054", 1800, 3, review_K=1, note="second population too small")
review("WT", "0068-0075", 1800, 2, review_status="rejected", note="weak spectrum")
change("# [review]", "[review]", section="[review]")
change("# table = ", "table = ", section="[review]")

In [ ]:
!hxex check hxex.toml
!hxex report hxex.toml --figures

In [ ]:
results = hxex.load_results("hxex.toml")  # again, with the review
hxex.show_series(results, "WT", "42-54", 2, kinds=["spectra"])

## Step 10: add the uncertainties

The bootstrap refits every spectrum many times to give each value a standard error. With one
and with two populations for every spectrum (`ks = [1, 2]`), which step 11 needs. This is the
slow step: 20 to 30 minutes on a laptop, a minute or two on a GPU.

In [ ]:
change("# [run.bootstrap]", "[run.bootstrap]", section="[run.bootstrap]")
change("# n_boot = 20", "n_boot = 20", section="[run.bootstrap]")
change('# ks = "selected"', "ks = [1, 2]", section="[run.bootstrap]")

In [ ]:
!hxex resume hxex.toml
!hxex report hxex.toml --figures

In [ ]:
results = hxex.load_results("hxex.toml")  # now with the bootstrap
hxex.show_series(results, "WT", "42-54", 2, kinds=["uptake"])

## Step 11: compare the variants with WT

`reference = "WT"` compares D109H and R120G with WT; `format = "png"` writes the figures as
pictures.

In [ ]:
change("# [analysis]", "[analysis]", section="[analysis]")
change('# reference = "WT"', 'reference = "WT"', section="[analysis]")
change('# format = "pdf"', 'format = "png"', section="[analysis]")

In [ ]:
!hxex analyse hxex.toml

R120G takes up more deuterium than WT over the first 80 residues, most at the early times:

In [ ]:
Image("results_analysis_WT/butterfly_R120G.png")

In [ ]:
Image("results_analysis_WT/woods_R120G_1min.png")

In [ ]:
Image("results_analysis_WT/compare_residues_1min.png")

The tables of the comparison are in `results_analysis_WT`: `compare_peptides.csv`,
`compare_residues.csv`, `species_uptake.csv`, and others. The guide's
[Outputs](https://github.com/tuttlelm/hxex/blob/main/docs/guide/outputs.md) page describes them.